In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader

# Define transforms - COMPLETE THE MISSING PARTS
transform = transforms.Compose([
    # TODO: Resize to 28x28
     transforms.Resize((28, 28)),
    transforms.Grayscale(3),  # Convert grayscale to RGB (Don't Touch!!)
    # TODO: Convert to Tensor
    transforms.ToTensor(),
    # TODO: Normalize with ImageNet mean=[0.485, 0.456, 0.406] and std=[0.229, 0.224, 0.225]
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

# Load EMNIST letters dataset (given)
train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset = EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)

# Note: EMNIST letters has labels 1-26 (A-Z), so we have 26 classes
num_classes = 26

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")

In [ ]:
# Letter mapping (labels are 1-26 for A-Z)
letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

# Create DataLoaders and display samples
# Write your code here
trainloader = DataLoader(train_dataset , batch_size=128, shuffle=True)
testloader = DataLoader(test_dataset, batch_size=100, shuffle=False)


import matplotlib.pyplot as plt
import numpy as np

mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

# Display 5 images
fig, axes = plt.subplots(1, 5, figsize=(15, 5))

imgs_indices = [270,233,110,89,15]

for i in range(5):
    img, label = train_dataset[imgs_indices[i]]

    # Convert tensor to numpy for visualization
    img_np = img.numpy().transpose(1, 2, 0)

    # Denormalize the image
    img_np = std * img_np + mean
    img_np = np.clip(img_np, 0, 1)

    # Show image
    axes[i].imshow(img_np)
    axes[i].axis('off')

plt.show()


In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_v2_s
from torchvision import datasets, transforms, models
# Write your code here
class  EfficientNetV2(nn.Module):
    def __init__(self, num_classes=26):
        super( EfficientNetV2, self).__init__()
        # Load pre-trained VGG16 model
        efficientnet = models.efficientnet_v2_s(weights=models.efficientnet_V2_S_Weights.IMAGENET1K_V1)

        # Freeze all parameters in the feature extractor
        for param in efficientnet.features.parameters():
            param.requires_grad = False

        self.features = efficientnet.features


In [ ]:
# Write your code here
import torch.optim as optim
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.classifier.parameters(), lr=0.001, momentum=0.9)

def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train() # Set the model to training mode
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad() # Zero the parameter gradients

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

    epoch_loss = running_loss / total_samples
    epoch_accuracy = correct_predictions / total_samples
    return epoch_loss, epoch_accuracy

def validate_epoch(model, dataloader, criterion, device):
    model.eval() # Set the model to evaluation mode
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    with torch.no_grad(): # Disable gradient calculation
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_samples += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()

    epoch_loss = running_loss / total_samples
    epoch_accuracy = correct_predictions / total_samples
    return epoch_loss, epoch_accuracy

In [ ]:
# Write your code here
num_epochs = 5

train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

print("Starting Training...")
for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, trainloader, criterion, optimizer, device)
    val_loss, val_acc = validate_epoch(model, testloader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    print(f'Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')

print("Finished Training.")

In [ ]:
# Write your code here
